# Lab 6: Your First Agent With the OpenAI Agents SDK

**IT3025: Introduction to Agentic AI**

Every lab through Module 6 built Study Buddy's agent loop by hand: a `while` loop that checked
`finish_reason`, a `for` loop over `tool_calls`, a dispatcher that matched `tool_call.function.name`
against your own tools, and a system prompt you curated yourself. That loop always worked the same
way no matter which tools you added, which is exactly why a framework can now take it over for you.

This lab rebuilds Study Buddy on top of the **OpenAI Agents SDK**, the framework this course uses
for the rest of the semester. Nothing about the *behavior* changes: the agent still has a persona,
still reads your study notes, and still has the same three tools Lab 5 gave it, one that writes a
log, one that calls an external API, and one that reads its own state back. What changes is who
writes the loop, and how each tool is described to the model.

**What you'll do, step by step:**
1. Install the OpenAI Agents SDK and this lab's other packages.
2. Load your Azure OpenAI credentials from `.env`, exactly as before.
3. Configure the SDK to run on Azure OpenAI instead of native OpenAI, and turn off tracing.
4. Load your study notes, same as Lab 3 and Lab 4.
5. Port all three of Lab 5's tools to `@function_tool`: `log_missed_question_tool`,
   `fetch_bonus_question_tool`, and `check_progress_tool`.
6. Build your first `Agent`: a name, instructions built from the persona and the notes, and all
   three tools.
7. Run it with `await Runner.run`.
8. Give it the same Gradio chat interface every lab since Lab 4 has used, so you can hold an actual
   conversation with it in your browser.
9. Try it end to end, and confirm all three tools actually run.

Run the cells in order, top to bottom.

## 0. Install the packages (run once)

This lab needs two new packages on top of what you already have: `openai-agents`, the OpenAI Agents
SDK itself, imported in Python as `agents`; and `requests`, needed by `fetch_bonus_question_tool` to
call the trivia API, the same way it did back in Lab 5. `gradio` is the same chat UI library you've
used since Lab 4, included here again for the interface at the end of this lab.

```
uv pip install --upgrade openai openai-agents pypdf python-dotenv requests gradio
```

Or run the equivalent cell below directly from the notebook.

In [ ]:
!uv pip install --upgrade openai openai-agents pypdf python-dotenv requests gradio

## 1. Imports

`AsyncAzureOpenAI` replaces the `AzureOpenAI` client you have used so far: the OpenAI Agents SDK is
built on `asyncio`, so it needs a client whose methods can be awaited. Everything else is either
familiar from earlier labs, or new and specific to the SDK:

- `Agent` — the agent itself: a name, instructions, and the tools it may use.
- `Runner` — runs the agent loop for you: calls the model, executes tools, and loops until there is
  a final answer.
- `function_tool` — a decorator that turns a plain, typed Python function into a tool, schema
  included, so you never hand write a JSON schema for it.
- `OpenAIChatCompletionsModel` — wraps a client so the SDK calls it using the Chat Completions API,
  which is what Azure OpenAI via APIM supports.
- `set_tracing_disabled` — turns off the SDK's built in tracing, which otherwise tries to send data
  to OpenAI's own platform using a real `OPENAI_API_KEY` that this course does not use.
- `gradio` (imported as `gr`) — the same chat UI library from Lab 4 and Lab 5, used again at the end
  of this lab to give Study Buddy a real chat window instead of typing into `input()` prompts.

`requests` and `html` are the same two imports `fetch_bonus_question_tool` needed back in Lab 5, for
calling the trivia API and cleaning up the HTML escaped text it returns. `typing.Literal` is new:
it is how you describe a fixed set of allowed string values to the SDK, the typed equivalent of the
`"enum"` key you wrote by hand in Lab 5's JSON schema for `fetch_bonus_question_tool`'s `category`
argument.

In [ ]:
from dotenv import load_dotenv
import os
import asyncio
import html
from typing import Literal, Optional

import requests
from openai import AsyncAzureOpenAI
from agents import Agent, Runner, function_tool, OpenAIChatCompletionsModel, set_tracing_disabled
from pypdf import PdfReader
import gradio as gr

## 2. Load your Azure OpenAI credentials from `.env`

Identical to every previous lab: same `.env` file, same gateway, same fixed deployment name.

In [ ]:
load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
DEPLOYMENT = "gpt-5.4-ptu"

print("Endpoint loaded:", bool(AZURE_OPENAI_ENDPOINT))
print("API key loaded:", bool(AZURE_OPENAI_API_KEY))

## 3. Configure the OpenAI Agents SDK for Azure OpenAI

Two things have to happen before you can build an `Agent`:

1. **Turn tracing off.** The SDK defaults to sending step by step traces to OpenAI's own platform,
   which requires a real `OPENAI_API_KEY`. This course only ever has Azure OpenAI credentials, so
   `set_tracing_disabled(True)` must be called before you build or run any agent.
2. **Build an async Azure client, and wrap it for the SDK.** `AsyncAzureOpenAI` is configured with
   the exact same three values you have used since Lab 3: your endpoint, your API key, and the API
   version. `OpenAIChatCompletionsModel` then wraps that client together with your deployment name,
   and that wrapped object is what you pass to `Agent` as its `model`.

In [ ]:
set_tracing_disabled(True)

azure_client = AsyncAzureOpenAI(
    azure_endpoint= AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2024-10-21",
)

azure_model = OpenAIChatCompletionsModel(
    model=DEPLOYMENT,
    openai_client=azure_client,
)

print("Azure model configured for the OpenAI Agents SDK.")

## 4. Load your study notes

Same as Lab 3 and Lab 4: put a `notes.pdf` file in this same folder, and this cell reads its text
in so it can be included in the agent's instructions below.

In [ ]:
persona = (
    "Study Buddy, an upbeat and encouraging AI study coach. Study Buddy keeps quizzes short and "
    "fun, genuinely celebrates every right answer, and never makes a student feel bad about "
    "getting something wrong - just curious to help them learn it."
)

reader = PdfReader("notes/notes.pdf")
notes = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        notes += text

print(persona)
print(notes[:300], "...")

## 5. Port Lab 5's three tools to `@function_tool`

Lab 5 gave Study Buddy three tools, each a plain Python function paired with a hand written JSON
schema dictionary describing its name, description, and parameters. Porting them to the SDK means
keeping each function's actual behavior identical and dropping the JSON schema entirely: one plain
Python function, with type hints and a docstring, decorated with `@function_tool`. The SDK builds
the schema for you from exactly those type hints and that docstring, so there is no separate JSON
to maintain, for any of the three.

### 5a. `log_missed_question_tool`

The simplest of the three: it takes one argument and writes a line to a file. Identical to Lab 4
and Lab 5, just decorated instead of paired with a JSON schema.

In [ ]:
LOG_PATH = "missed_questions.txt"

@function_tool
def log_missed_question_tool(question: str) -> str:
    """Record a study question the student got wrong or was unsure about, so they can review it later.

    Args:
        question: The exact question the student missed or was unsure about.
    """
    print(f"Tool called to log a missed question: {question}")
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(question + "\n")
    return "Missed question logged"

print("Tool ready:", log_missed_question_tool.name)

### 5b. `fetch_bonus_question_tool`

[Open Trivia DB](https://opentdb.com/api_config.php) is the same free, no key trivia API Lab 5
used: one `GET` request to `opentdb.com/api.php` returns a ready made multiple choice question as
JSON. The `category` argument is optional, and `Literal["general_knowledge", "science_computers"]`
is the typed way of saying exactly what Lab 5's JSON schema said with `"enum": [...]`: the model may
only pass one of these two strings, or leave the argument out entirely.

The same note from Lab 5 still applies: this is a free, shared, best effort service with no uptime
guarantee. The `try`/`except` below hands back a short, friendly string instead of crashing the
notebook if the API is slow or briefly unavailable.

In [ ]:
OPENTDB_CATEGORIES = {"general_knowledge": 9, "science_computers": 18}

@function_tool
def fetch_bonus_question_tool(category: Optional[Literal["general_knowledge", "science_computers"]] = None) -> str:
    """Fetch a fun bonus trivia question from an external trivia database, for when the student asks for a bonus question or a break from the study notes.

    Args:
        category: Which trivia category to pull a bonus question from, if the student asked for one.
    """
    print(f"Tool called to fetch a bonus trivia question (category={category})")
    params = {"amount": 1, "type": "multiple"}
    if category in OPENTDB_CATEGORIES:
        params["category"] = OPENTDB_CATEGORIES[category]

    try:
        response = requests.get("https://opentdb.com/api.php", params=params, timeout=20)
        data = response.json()
    except (requests.exceptions.RequestException, ValueError) as e:
        print(f"fetch_bonus_question_tool network error: {e}")
        return "The trivia API is temporarily unavailable - just skip the bonus question and continue the quiz."

    if data.get("response_code") != 0 or not data.get("results"):
        return "Could not fetch a bonus question right now - just skip it and continue the quiz."

    q = data["results"][0]
    question = html.unescape(q["question"])
    answer = html.unescape(q["correct_answer"])
    return f"Bonus question: {question} (correct answer: {answer})"

print("Tool ready:", fetch_bonus_question_tool.name)

### 5c. `check_progress_tool`

The simplest of the three to call, since it takes no arguments at all: it reads the
`missed_questions.txt` file that `log_missed_question_tool` already writes, and reports how many
questions are logged in it. A useful contrast between the three: one writes a file, one calls an
external API, and this one reads a file back to answer a question about the student's own
progress.

In [ ]:
@function_tool
def check_progress_tool() -> str:
    """Check how many study questions the student has missed so far, for when they ask about their progress."""
    print("Tool called to check quiz progress")
    try:
        with open(LOG_PATH, encoding="utf-8") as f:
            missed = [line for line in f if line.strip()]
    except FileNotFoundError:
        missed = []

    if not missed:
        return "No questions have been missed yet - a clean sheet so far!"
    return f"So far, {len(missed)} question(s) have been logged as missed."

print("Tool ready:", check_progress_tool.name)

## 6. Build the Agent

This is the declarative version of everything Lab 5's system prompt and dispatcher did by hand:

- `name` is just a label for the agent, useful in tracing and logs.
- `instructions` is exactly what used to be your system prompt: the persona, the notes, and the
  rules about when to use each tool.
- `tools` is the list of `@function_tool` decorated functions the agent may call, all three of them.
- `model` is the Azure wrapped model object you built in step 3.

No `while` loop, no `for tool_call in message_obj.tool_calls`, and no `if name == ...` dispatcher
picking between three tools by hand: `Agent` only declares what the agent is, not how the loop
runs, no matter how many tools it has.

In [ ]:
instructions = f"""
# Your role

You are {persona}

You quiz a student on the study notes below.

# Context

Here are the study notes you are quizzing the student on:

{notes}

# Rules

Ask exactly ONE question at a time, based only on the notes above. Wait for the student's answer
before asking the next question.

After the student answers, say clearly whether they were right or wrong, and give a one-sentence
explanation either way, staying in character the whole time.

If the student's answer was wrong, or they say "I don't know", you must record it as a missed
question using log_missed_question_tool before moving on to the next question.

If the student asks for a bonus question, or seems to want a break from the notes, use
fetch_bonus_question_tool to get one - this question does not need to relate to the notes.

If the student asks how they are doing, or asks for a progress check, use check_progress_tool to
tell them how many questions they've missed so far.

IMPORTANT: only ask quiz questions from the notes above. If the student asks about something
outside the notes (and it isn't a bonus question or a progress check), say that it isn't covered
in this study set.
"""

study_buddy = Agent(
    name="Study Buddy",
    instructions=instructions,
    tools=[log_missed_question_tool, fetch_bonus_question_tool, check_progress_tool],
    model=azure_model,
)

print("Agent built:", study_buddy.name)

## 7. Run it: `await Runner.run`

`Runner` replaces the entire loop from earlier labs. `await Runner.run(agent, input)` is the native
async entry point: it calls the model, runs any tool the model asks for, and loops until there is a
final answer. Jupyter notebooks support top level `await`, so you can call it directly in a cell,
as long as you use `await`.

The SDK also ships a synchronous wrapper, `Runner.run_sync`, for use outside notebooks. Skip it
here: Jupyter already runs its own event loop in the background, and calling `Runner.run_sync` from
inside a notebook cell raises `RuntimeError: AgentRunner.run_sync() cannot be called when an event
loop is already running.` Always use `await Runner.run(...)` in a notebook.

In [ ]:
result = await Runner.run(study_buddy, "Hi! Can you quiz me with one question from my notes?")
print(result.final_output)

In [ ]:
result = await Runner.run(study_buddy, "Actually, I don't know the answer to that one.")
print(result.final_output)

Check `missed_questions.txt` in this folder: if the model decided the student's last message
counted as a missed question, `log_missed_question_tool` should have appended a line to it, exactly
like it did in Lab 5, just without you writing the dispatcher that routed the call there.

In [ ]:
try:
    with open(LOG_PATH, encoding="utf-8") as f:
        print(f.read())
except FileNotFoundError:
    print("No missed questions logged yet.")

Now try the other two tools the same way. Each should look exactly like the calls above, since
`Runner.run` treats all three tools identically, it has no idea one of them makes a network call and
another just reads a file.

In [ ]:
result = await Runner.run(study_buddy, "Can I get a bonus question, science and computers category?")
print(result.final_output)

In [ ]:
result = await Runner.run(study_buddy, "How am I doing so far?")
print(result.final_output)

## 8. Give it a real chat UI with Gradio

Every lab since Lab 4 has ended the same way: a plain Python `chat(message, history)` function,
handed to `gr.ChatInterface`, which draws the actual chat window, keeps the visible conversation on
screen, and calls your function again every time the student sends a message. None of that changes
here. What changes is what happens *inside* `chat`.

One detail makes this easy: **`gr.ChatInterface` accepts an `async def` function directly.** You do
not need to wrap anything in `asyncio.run` yourself; Gradio awaits your function for you, so `chat`
can call `await Runner.run(...)` exactly the way earlier cells in this notebook already did.

Gradio's `history` comes in as a list of `{"role": ..., "content": ...}` dictionaries, the same
shape `Runner.run` expects as its input list. `chat` rebuilds that list with a plain comprehension,
appends the new `message` as one more `user` turn, and hands the whole thing to `Runner.run`, the
same pattern you used two cells ago to test the agent by hand.

`chat` returns `result.final_output`, a plain string, and `gr.ChatInterface` takes care of showing
it in the window. Nothing about calling a tool changes either: whichever of the three tools the
model decides to call mid conversation, `Runner.run` still runs it for you, exactly as it did when
you tested the agent directly above.

In [ ]:
async def chat(message, history):
    history = [{"role": item["role"], "content": item["content"]} for item in history]
    result = await Runner.run(study_buddy, history + [{"role": "user", "content": message}])
    return result.final_output

gr.ChatInterface(chat).launch(inbrowser=True)

Try a conversation that would trigger all three tools across a few turns, a missed answer, a
request for a bonus question, and a progress check, then interrupt the cell (stop button) when you
are done chatting, the same way you did in earlier labs.

## Recap

- `Agent` replaced your system prompt and tool list.
- `@function_tool` replaced three hand written JSON schemas, one per tool, with three decorated
  Python functions.
- `await Runner.run` replaced your entire `while` / `for` agent loop and the `if name == ...`
  dispatcher that routed between your three tools. Stick to `await Runner.run(...)` in notebooks;
  `Runner.run_sync` conflicts with Jupyter's own event loop.
- `OpenAIChatCompletionsModel` plus `AsyncAzureOpenAI` replaced a plain `AzureOpenAI` client, so the
  same Azure OpenAI via APIM setup you've used all semester still works.
- `set_tracing_disabled(True)` is the one line that only matters because this course has no native
  OpenAI key.
- `gr.ChatInterface` already accepted an async `chat` function with no changes needed, and its
  `history` format already matches what `Runner.run` expects.

Everything the agent actually does is identical to Lab 5, same persona, same notes, same three
tools. What changed is who wrote the loop, and how much code it took to describe each tool.